Esse notebook é o processamento dos dados da base_dados_cadastrais para a camada Trusted na AWS

# Iniciando o Spark

In [20]:
!pip install pyspark

In [21]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Trusted_base_dados_cadastrais") \
    .config('spark.ui.port', '4050') \
    .getOrCreate()

# Importando bibliotecas

In [4]:
import os
import sys
import time
import datetime
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, udf, input_file_name
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Funções

In [6]:
#Criar uma função log para registrar a data
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') +' >>>'
dt_proc = datetime.now().strftime('%Y%m%d%H%M%S')
current_date = datetime.now().strftime('%Y%m%d')

# Importando os dados

In [22]:
raw_path = "s3://meu-bucket/raw/base_dados_cadastrais"


In [23]:
df_base_dados_cadastrais = spark.read.parquet(raw_path)
df_base_dados_cadastrais.createOrReplaceTempView("df_Trusted_base_dados_cadastrais")

# Processamento

In [24]:
#Alterando o datatype da base
df_base_dados_cadastrais = spark.sql(f"""
    SELECT

        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        '{dt_proc}' AS dt_Proc,
        CAST(SAFRA AS INT) AS SAFRA,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS SAFRA_ANO,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS SAFRA_MES,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS FLAG_INSTALACAO,
        CAST(FPD AS BOOLEAN) AS FPD,
        CAST(PROD AS STRING) AS PROD,
       CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(STATUSRF AS STRING) AS STATUSRF,
        to_date(DATADENASCIMENTO, 'dd/MM/yyyy') AS DATA_DE_NASCIMENTO,
        to_date(var_12, 'dd/MM/yyyy') AS var_12,

        CAST(var_02 AS INT) AS var_02,
        CAST(var_03 AS INT) AS var_03,
        CAST(var_04 AS INT) AS var_04,
        CAST(var_05 AS INT) AS var_05,
        CAST(var_06 AS INT) AS var_06,
        CAST(var_07 AS FLOAT) AS var_07,
        CAST(var_08 AS INT) AS var_08,
        CAST(var_09 AS INT) AS var_09,
        CAST(var_10 AS STRING) AS Profissao,
        CAST(var_11 AS FLOAT) AS var_11,
        CAST(var_13 AS STRING) AS var_13,
        CAST(var_14 AS INT) AS var_14,
        CAST(var_15 AS STRING) AS Estado,
        CAST(var_16 AS INT) AS var_16,
        CAST(var_17 AS INT) AS var_17,
        CAST(var_18 AS STRING) AS var_18,
        CAST(var_19 AS STRING) AS var_19,
        CAST(var_20 AS STRING) AS var_20,
        CAST(var_21 AS STRING) AS var_21,
        CAST(var_22 AS STRING) AS Cargo,
        CAST(var_23 AS STRING) AS var_23,
        CAST(var_24 AS STRING) AS var_24,
        CAST(var_25 AS STRING) AS Tipo_de_Auxilio,
        CAST(CEP_3_digitos AS STRING) AS CEP_3_digitos

    FROM df_Trusted_base_dados_cadastrais

""")

In [ ]:
# Salvar tabela no bucket na camada Trusted
df_base_dados_cadastrais.write \
    .mode("append") \
    .partitionBy("SAFRA") \
    .parquet("s3://meu-bucket/minha-pasta/df_base_dados_cadastrais")


In [18]:
#Consultando a tabela salva
df_base_dados_cadastrais = spark.read.parquet("s3://meu-bucket/minha-pasta/df_base_dados_cadastrais")
df_base_dados_cadastrais.createOrReplaceTempView("df_Trusted_base_dados_cadastrais")
df_base_dados_cadastrais.show()


+-----------+--------------+---------+---------+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+--------+------+------+---------+------+----------+------+------+------+------+----------+--------+------+------------+------------+-------------+----------+--------------------+-------------+------+
|    NUM_CPF|       dt_Proc|SAFRA_ANO|SAFRA_MES|FLAG_INSTALACAO|  FPD|PROD|flag_mig2|STATUSRF|DATA_DE_NASCIMENTO|    var_12|var_02|var_03|var_04|var_05|var_06|  var_07|var_08|var_09|Profissao|var_11|    var_13|var_14|Estado|var_16|var_17|    var_18|  var_19|var_20|      var_21|       Cargo|       var_23|    var_24|     Tipo_de_Auxilio|CEP_3_digitos| SAFRA|
+-----------+--------------+---------+---------+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+--------+------+------+---------+------+----------+------+------+------+------+----------+--------+------+------------+----